# AMBULANCE ROUTING AND EMERGENCY MANAGEMENT DIGITAL TWIN - Prototype

## TEAM MEMBERS -
### Aditya Pandey - A04
### Arshiya Pandey - A26
### Aryaman Verma - A29
### Nayan Kumbhare - A56

In [ ]:
!pip install -q osmnx networkx pandas numpy folium

import os
import math
import random
import numpy as np
import pandas as pd
import networkx as nx
import folium

from IPython.display import display

random.seed(39)

print("Libraries loaded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 995.1 kB/s eta 0:00:00
Libraries loaded successfully!


In [ ]:
# ==========================================
# DIGITAL TWIN - ROAD NETWORK
# ==========================================

G = nx.grid_2d_graph(6, 6)

# Convert (row, col) into simple node IDs
mapping = {}
for i, node in enumerate(G.nodes()):
    mapping[node] = i

G = nx.relabel_nodes(G, mapping)

# Add coordinates
lat0 = 21.1458
lon0 = 79.0882

for node in G.nodes():

    row = node // 6
    col = node % 6

    G.nodes[node]["lat"] = lat0 + row * 0.003
    G.nodes[node]["lon"] = lon0 + col * 0.003

# Add road properties
for u, v in G.edges():

    G[u][v]["distance"] = random.randint(300, 600)
    G[u][v]["speed"] = 40

    # Base travel time
    G[u][v]["base_time"] = (
        G[u][v]["distance"] / 1000
    ) / G[u][v]["speed"] * 60

    # Traffic state
    G[u][v]["traffic"] = "Normal"
    G[u][v]["traffic_factor"] = 1.0

print("Digital Twin created!")
print("Road nodes:", len(G.nodes))
print("Road connections:", len(G.edges))

Digital Twin created!
Road nodes: 36
Road connections: 60


In [ ]:
# ==========================================
# SIMULATED TRAFFIC
# ==========================================

traffic_factors = {
    "Normal": 1.0,
    "Moderate": 1.5,
    "Heavy": 2.5,
    "Blocked": float("inf")
}

edges = list(G.edges())

# Some roads become congested
for u, v in edges[:5]:

    G[u][v]["traffic"] = "Heavy"
    G[u][v]["traffic_factor"] = traffic_factors["Heavy"]


for u, v in edges[5:8]:

    G[u][v]["traffic"] = "Moderate"
    G[u][v]["traffic_factor"] = traffic_factors["Moderate"]


# One road blocked
u, v = edges[8]

G[u][v]["traffic"] = "Blocked"
G[u][v]["traffic_factor"] = float("inf")


print("Traffic simulation applied.")

for u, v in edges[:10]:

    print(
        f"Road {u} → {v}: "
        f"{G[u][v]['traffic']}"
    )

Traffic simulation applied.
Road 0 → 6: Heavy
Road 0 → 1: Heavy
Road 1 → 7: Heavy
Road 1 → 2: Heavy
Road 2 → 8: Heavy
Road 2 → 3: Moderate
Road 3 → 9: Moderate
Road 3 → 4: Moderate
Road 4 → 10: Blocked
Road 4 → 5: Normal


In [ ]:
# ==========================================
# AMBULANCES
# ==========================================

ambulances = pd.DataFrame({

    "ambulance_id": [
        "A01",
        "A02",
        "A03",
        "A04"
    ],

    "node": [
        0,
        10,
        25,
        30
    ],

    "status": [
        "Available",
        "Available",
        "Busy",
        "Available"
    ]
})

display(ambulances)

,ambulance_id,node,status
0,A01,0,Available
1,A02,10,Available
2,A03,25,Busy
3,A04,30,Available


In [ ]:
# ==========================================
# HOSPITALS
# ==========================================

hospitals = pd.DataFrame({

    "hospital_id": [
        "H01",
        "H02",
        "H03"
    ],

    "node": [
        5,
        20,
        35
    ],

    "facility": [
        "Emergency",
        "Emergency",
        "General"
    ],

    "beds_available": [
        12,
        4,
        18
    ]
})

display(hospitals)

,hospital_id,node,facility,beds_available
0,H01,5,Emergency,12
1,H02,20,Emergency,4
2,H03,35,General,18


In [ ]:
# ==========================================
# EMERGENCY
# ==========================================

emergency = {

    "id": "E01",

    "node": 35,

    "priority": "HIGH",

    "required_facility": "Emergency"
}

print("Emergency Generated")
print(emergency)

Emergency Generated
{'id': 'E01', 'node': 35, 'priority': 'HIGH', 'required_facility': 'Emergency'}


In [ ]:
# ==========================================
# TRAFFIC-AWARE DIJKSTRA
# ==========================================

def traffic_weight(u, v, data):

    if data["traffic"] == "Blocked":
        return float("inf")

    return (
        data["base_time"]
        * data["traffic_factor"]
    )


def find_route(source, destination):

    H = nx.Graph()

    for u, v, data in G.edges(data=True):

        weight = traffic_weight(u, v, data)

        if weight != float("inf"):

            H.add_edge(
                u,
                v,
                weight=weight
            )

    path = nx.shortest_path(
        H,
        source,
        destination,
        weight="weight"
    )

    travel_time = nx.shortest_path_length(
        H,
        source,
        destination,
        weight="weight"
    )

    distance = 0

    for a, b in zip(path[:-1], path[1:]):

        distance += G[a][b]["distance"]

    return path, distance, travel_time

In [ ]:
# ==========================================
# AMBULANCE SELECTION
# ==========================================

results = []

for _, ambulance in ambulances.iterrows():

    if ambulance["status"] != "Available":
        continue

    try:

        path, distance, time = find_route(
            ambulance["node"],
            emergency["node"]
        )

        results.append({

            "Ambulance":
                ambulance["ambulance_id"],

            "Distance_m":
                round(distance, 2),

            "ETA_min":
                round(time, 2),

            "Route":
                path
        })

    except nx.NetworkXNoPath:

        pass


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "ETA_min"
)

display(
    results_df[
        [
            "Ambulance",
            "Distance_m",
            "ETA_min"
        ]
    ]
)

best = results_df.iloc[0]

print("\n SELECTED AMBULANCE")
print("====================")
print("Ambulance:", best["Ambulance"])
print("Distance:", best["Distance_m"], "m")
print("ETA:", best["ETA_min"], "minutes")
print("Route:", best["Route"])

,Ambulance,Distance_m,ETA_min
2,A04,2045,3.07
1,A02,2065,3.10
0,A01,3781,6.59



 SELECTED AMBULANCE
Ambulance: A04
Distance: 2045 m
ETA: 3.07 minutes
Route: [30, 31, 32, 33, 34, 35]


In [ ]:
# ==========================================
# HOSPITAL SELECTION
# ==========================================

hospital_results = []

for _, hospital in hospitals.iterrows():

    if hospital["facility"] != emergency["required_facility"]:
        continue

    if hospital["beds_available"] <= 0:
        continue

    try:

        path, distance, time = find_route(
            emergency["node"],
            hospital["node"]
        )

        hospital_results.append({

            "Hospital":
                hospital["hospital_id"],

            "Beds":
                hospital["beds_available"],

            "Distance_m":
                round(distance, 2),

            "ETA_min":
                round(time, 2),

            "Route":
                path
        })

    except nx.NetworkXNoPath:

        pass


hospital_df = pd.DataFrame(
    hospital_results
).sort_values("ETA_min")

display(
    hospital_df[
        [
            "Hospital",
            "Beds",
            "Distance_m",
            "ETA_min"
        ]
    ]
)

best_hospital = hospital_df.iloc[0]

print("\n RECOMMENDED HOSPITAL")
print("======================")
print(
    "Hospital:",
    best_hospital["Hospital"]
)

print(
    "ETA:",
    best_hospital["ETA_min"],
    "minutes"
)

print(
    "Beds:",
    best_hospital["Beds"]
)

,Hospital,Beds,Distance_m,ETA_min
1,H02,4,2027,3.04
0,H01,12,2381,3.57



 RECOMMENDED HOSPITAL
Hospital: H02
ETA: 3.04 minutes
Beds: 4


In [ ]:
# ==========================================
# DIGITAL TWIN MAP
# ==========================================

def coordinates(node):

    return [
        G.nodes[node]["lat"],
        G.nodes[node]["lon"]
    ]


# Map center
center = coordinates(18)

m = folium.Map(
    location=center,
    zoom_start=15
)


# ------------------------------------------
# ROADS
# ------------------------------------------

for u, v, data in G.edges(data=True):

    road_color = {

        "Normal": "green",

        "Moderate": "orange",

        "Heavy": "red",

        "Blocked": "black"

    }[data["traffic"]]

    folium.PolyLine(

        [
            coordinates(u),
            coordinates(v)
        ],

        color=road_color,

        weight=5,

        tooltip=(
            f"Traffic: "
            f"{data['traffic']}"
        )

    ).add_to(m)


# ------------------------------------------
# AMBULANCES
# ------------------------------------------

for _, ambulance in ambulances.iterrows():

    color = (
        "blue"
        if ambulance["status"] == "Available"
        else "gray"
    )

    folium.Marker(

        coordinates(
            ambulance["node"]
        ),

        tooltip=(
            f"Ambulance "
            f"{ambulance['ambulance_id']} "
            f"| {ambulance['status']}"
        ),

        icon=folium.Icon(
            color=color,
            icon="plus"
        )

    ).add_to(m)


# ------------------------------------------
# HOSPITALS
# ------------------------------------------

for _, hospital in hospitals.iterrows():

    folium.Marker(

        coordinates(
            hospital["node"]
        ),

        tooltip=(
            f"{hospital['hospital_id']} "
            f"| Beds: "
            f"{hospital['beds_available']}"
        ),

        icon=folium.Icon(
            color="purple",
            icon="home"
        )

    ).add_to(m)


# ------------------------------------------
# EMERGENCY
# ------------------------------------------

folium.Marker(

    coordinates(
        emergency["node"]
    ),

    tooltip=(
        "🚨 Emergency E01 | HIGH"
    ),

    icon=folium.Icon(
        color="red",
        icon="warning-sign"
    )

).add_to(m)


# ------------------------------------------
# SELECTED AMBULANCE ROUTE
# ------------------------------------------

route_coordinates = [

    coordinates(node)

    for node in best["Route"]

]

folium.PolyLine(

    route_coordinates,

    color="blue",

    weight=8,

    tooltip="Recommended Ambulance Route"

).add_to(m)


display(m)